# Linear regression-based cache eviction policy

### Does a linear regression model provide powerful cache eviction decisions?

In this notebook we implement the same pipeline (data preparation, data generation, validation, training, and testing) for a **Linear Regression** model as we did for the LSTM, using the same dataset and evaluation metrics.

*What is our goal?* We aim at comparing a complex neural network against a simpler baseline to determine if a linear approach is sufficient for cache eviction decisions.

*What do we expect?* While Linear Regression is computationally efficient, it is likely too simplistic for this task. It lacks the capacity to capture non-linear access patterns or complex temporal dependencies. Consequently, we expect it to underperform compared to the LSTM, which is specifically designed to learn midterm and long-term dependencies within sequence data.

---

### Configuration

In [ ]:
from components.yaml.io.loader import load_yaml
from pipeline.const import DATASET_PROCESSED_TYPE, PIPELINE_CONFIG_FILE_PATH

# Load pipeline configuration object
pipeline_config = load_yaml(PIPELINE_CONFIG_FILE_PATH)

# Define configuration for Logistic Regression
lr_config = {
    "dataset": {"test_size": 0.8},
    "model": {
        "class_weight": "balanced",
    },
    "validation": {
        "folds": 5,
        "scoring": "f1_weighted",
        "search_space": {
            "C": [0.001, 0.01, 0.1, 1, 10, 100],
            "max_iter": [1000, 2000],
            "penalty": ["l1", "l2"],
            "solver": ["saga"],
        },
        "shuffle": False,
        "verbose": 3,
    },
}

---

### 1. Data Preparation and Data Preprocessing

These steps aim to generate and/or exploring data, as well as process them by removing missing values and building the following features:
- Cosine time (`cos_time`)
- Sine time (`sin_time`)
- Local frequency (`local_frequency`)
- Local recency (`local_recency`)

The first two features are used for encoding time trigonometrically for a better temporal representation. The latter two are designed to capture access patterns by summarizing historical usage information.

In [ ]:
from pipeline.steps.data_preparer import prepare_data
from pipeline.steps.data_preprocessor import preprocess_data

# Data preparation and processing steps
prepare_data()
preprocess_data()

---

### 2. Validation and Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    TimeSeriesSplit,
    train_test_split,
)

from components.dataset.io.loader import load_dataset
from components.dataset.io.locator import get_dataset_abs_path

# Define Time Series Split object
tscv = TimeSeriesSplit(n_splits=lr_config["validation"]["folds"])

# Define a logistic regression model
lr = LogisticRegression(
    class_weight=lr_config["model"]["class_weight"],
)

# Define Grid Search Cross Validation object
grid_search = GridSearchCV(
    estimator=lr,
    param_grid=lr_config["validation"]["search_space"],
    cv=tscv,
    scoring=lr_config["validation"]["scoring"],
    verbose=lr_config["validation"]["verbose"],
)

# Load processed data
df = load_dataset(
    get_dataset_abs_path(
        DATASET_PROCESSED_TYPE,
        pipeline_config["data"]["general"]["mode"],
    ),
)

# Extract features and targets from data
X = df[["cos_time", "sin_time", "local_frequency", "local_recency"]].values
y = df["request"].values

# Split train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=lr_config["dataset"]["test_size"],
    shuffle=lr_config["validation"]["shuffle"],
)

# Run grid search
grid_search.fit(X_train, y_train)

### 3. Testing

In [ ]:
from components.evaluation.model.metrics.calculator import (
    calculate_model_metrics,
)

# Use the best model for making predictions
# over the testing set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Calculate and show evaluation metrics
lr_metrics = calculate_model_metrics(
    y_test,
    y_pred,
)
print(lr_metrics)